In [1]:
from ipyleaflet import Map, CircleMarker, basemaps, Polyline
from _main import curvecreator
from mainhouse import curvecreatorh
from IPython.display import display
import buildcache
import geopandas as gpd
from shapely.geometry import Point
from ipywidgets import HBox
from gui import create_gui
from stationrating import calc_station_rating

def jupmain():
    
    #initialize map
    m = Map(
        center=(46.4983, 11.3548),
        zoom=13,
        basemap=basemaps.OpenStreetMap.Mapnik,
        layout={'width': '100%', 'height': '730px'},#100% for height breaks jupiternotebook
        scroll_wheel_zoom=True
    )
    
    #load gui from separate file
    gui = create_gui()
    has_to_be_fast = gui["has_to_be_fast"]
    big_nimby = gui["big_nimby"]
    big_yimby = gui["big_yimby"]
    has_to_be_fast_text = gui["has_to_be_fast_text"]
    big_nimby_text = gui["big_nimby_text"]
    big_yimby_text = gui["big_yimby_text"]
    reset_button = gui["reset_button"]
    status = gui["status"]
    show_circle = gui["show_circle"]
    show_blue_line = gui["show_blue_line"]
    show_temp_line = gui["show_temp_line"]
    make_stations = gui["make_stations"]
    ui = gui["ui"]

    
    #optional reset logic, connected to gui button
    def on_reset_clicked(b):
        nonlocal utm_crs
        polist.clear()
        polist_wgs84.clear()
        utm_crs = None

        for layer in list(m.layers)[1:]:
            m.remove_layer(layer)

        #with out:
        #    out.clear_output()
        #    print("Reset complete.")

    reset_button.on_click(on_reset_clicked)
   
    #this is the best coordinate system to use and is based on the location of the first click
    utm_crs = None

    #initialize polist
    polist = []         #(x, y) in lokalem UTM, Meter
    polist_wgs84 = []   #(lat, lon) in EPSG:4326    
    
    def handle_click(**kwargs):
        nonlocal utm_crs#, buildings_displayed
    
        if kwargs.get("type") != "click":
            return
    
        lat, lon = kwargs.get("coordinates")
        polist_wgs84.append((lat, lon))
        
        point_wgs84 = gpd.GeoSeries(
            [Point(lon, lat)],
            crs="EPSG:4326"
        )

        #determine local UTM zone from first click
        if utm_crs is None:
            utm_crs = point_wgs84.estimate_utm_crs()

        point_utm = point_wgs84.to_crs(utm_crs).iloc[0]
        polist.append((point_utm.x, point_utm.y))
    
        #show clicked points
        m.add_layer(
            CircleMarker(
                location=(lat, lon),
                radius=3,
                color="blue",
                fill_color="aqua",
                fill_opacity=0.1,
                opacity=0.2
            )
        )
        #load text values into sliders
        angle_weight = has_to_be_fast_text.value
        first_value = big_nimby_text.value
        second_value = big_yimby_text.value
                
        #create line and cache on fourth click
        if len(polist) % 4 == 0:
    
            ultquatr = polist[-4:]
    
            point2_wgs84 = polist_wgs84[-3]
            point3_wgs84 = polist_wgs84[-2]
            status.value = "<b style='color: orange;'>Lade Gebäude aus OSM...</b>"

            mid_lat, mid_lon, radius = buildcache.building_cache(
                point2_wgs84,
                point3_wgs84,
                utm_crs
            )
            #show circle if desired
            if show_circle.value:
                from ipyleaflet import Circle
                circle = Circle(
                    location=(mid_lat, mid_lon),
                    radius=int(radius),
                    color="purple",
                    weight=1,
                    fill=True,
                    fill_color="purple",
                    fill_opacity=0.1
                )
                m.add_layer(circle)
            
    
            #######curve creation part######
            if show_blue_line.value:
                status.value = "<b style='color: blue;'>Erstelle Kurve...</b>"
                #angular curve
                lineplot = curvecreator(ultquatr.copy())
                #convert to WGS84 for display
                lineplot_wgs84 = gpd.GeoSeries(
                    [Point(x, y) for x, y in lineplot],
                    crs=utm_crs
                ).to_crs("EPSG:4326")
                
                line = Polyline(
                    locations=[(geom.y, geom.x) for geom in lineplot_wgs84],
                    color="blue",
                    weight=3,
                    fill=False
                )
                m.add_layer(line)
            status.value = "<b style='color: #85082b;'>Erstelle Trasse...</b>"
            
            #house & angular curve
            lineploth, station_polist = curvecreatorh(
                ultquatr.copy(),
                buildcache.building_gdf_local,
                angle_weight,
                first_value,
                second_value,
                status,
                radius,
                m,
                utm_crs,
                show_temp_line.value
            )      
            
            #calls station rating function and generates statins
            #station_rating = None
            if make_stations.value:
                status.value = "<b style='color: lime;'>Finde Haltepunkte...</b>"
                station_points = []
                #creates stations
                if station_polist is not None:
                    station_points = calc_station_rating(
                        station_polist,
                        buildcache.building_gdf_local,
                        search_radius=800
                    )
    
                    #plot station points on the map
                    if station_points:
                        station_points_wgs84 = gpd.GeoSeries(
                            [Point(x, y) for x, y in station_points],
                            crs=utm_crs
                        ).to_crs("EPSG:4326")
    
                        for geom in station_points_wgs84:
                            m.add_layer(
                                CircleMarker(
                                    location=(geom.y, geom.x),
                                    radius=6,
                                    color="green",
                                    fill_color="lime",
                                    fill_opacity=0.8
                                )
                            )
            #generates last line if tempdisplay is of
            if not show_temp_line.value:
                #convert to WGS84 for display
                lineploth_wgs84 = gpd.GeoSeries(
                    [Point(x, y) for x, y in lineploth],
                    crs=utm_crs
                ).to_crs("EPSG:4326")
                
                lineh = Polyline(
                    locations=[(geom.y, geom.x) for geom in lineploth_wgs84],
                    color="#85082b",
                    weight=3,
                    fill=False
                )
                m.add_layer(lineh)
            status.value = "<b style='color: green;'>Fertig!</b>"


    #attach click handler
    m.on_interaction(handle_click)
    
    display(HBox(
        [m, ui],
    ))

jupmain()